# RF Board — PYNQ Main Notebook

Direct hardware control on the Zynq-7020 ARM PS (no TCP).

**Sequence** (mirrors `PC_code/main.py`):
1. Load PL bitstream overlay
2. Instantiate `PynqController`
3. Read ADC chip identification
4. Configure CDCE62005 clock distributor
5. Initialize DACs with default settings
6. **BRAM profile control** — init, write frequency profiles, write amplitude ramp profiles
7. (Optional) Configure NCO frequencies via PL DDS

In [65]:
# Step 1 — Load the PL bitstream overlay
from pynq import Overlay
import time
from pynq_controller import PynqController

ol = Overlay('./DAC1ST125MHz_DAC2MT125MHz.xsa')
print('Overlay loaded')

BOARD_VER = 2
zynq = PynqController(ol, board_ver=BOARD_VER, enable_bram=True, enable_axi=True)

### Supply DACCLK and OSTR
CDCE_CONFIG_no_FPGA = '8340032083400321EA040302811C0303EA84031410000BE504BE09F6BD0037F7' # DACCLK1 DACCLK2 1 GHz, OSTR effective 15.625 MHz, FPGA_CLK ADC_CLK off
zynq.configure_clock(CDCE_CONFIG_no_FPGA,debug=1)


Overlay loaded

[PynqController] Initializing (board v2)...
CDCE62005 ready (board v2): CLK=MIO12, MOSI=MIO10, MISO=MIO11, LE=MIO15
  CS_DAC1=MIO13, CS_DAC2=MIO14, CS_ADC=MIO0 — all deasserted
SPI3WireBus ready (board v2): CLK=MIO12, SDIO=MIO11
DAC3484 DAC1 ready (CS_DAC1)
DAC3484 DAC2 ready (CS_DAC2)
AD9643 ADC ready
  MT_CTRL   GPIO@0x41210000
BRAMController ready
  DDS clock : 250.0 MHz
  CH1  freq@0x44000000  amp@0x40000000
  CH2  freq@0x46000000  amp@0x42000000
✓ PynqController ready

Configuring CDCE62005...
  reg[0] <- 0x8340032
  reg[1] <- 0x8340032
  reg[2] <- 0xEA04030
  reg[3] <- 0x811C030
  reg[4] <- 0xEA84031
  reg[5] <- 0x10000BE
  reg[6] <- 0x04BE09F
  reg[7] <- 0xBD0037F
  reg[8] = 0x20009BF (OK)
Done.


True

## Configure DAC1 (Single Tone) and DAC2 (Multi Tone-Brams): 
#### Toggle RESETB

In [66]:
# ___________________________________________________________________________________________________________________
#--------------------------------------------- DAC2 Initialization- Part 1 ------------------------------------------
### **Program SIF registers:**
zynq.dac2.write_register(0x00, 0x089F) # INTERP = 16, fifo on, clkdiv_sync_ena invsincAB+CD
zynq.dac2.write_register(0x01, 0x050E)
zynq.dac2.write_register(0x02, 0xF052) # 16bit_in, mixer_ena, nco_ena, twos_comp
zynq.dac2.write_register(0x03, 0xF000)
zynq.dac2.write_register(0x07, 0xD8FF) # un-mask FIFO collison, DACCLK-gone, DATACLK-gone

# # registers 0x08-0x11 are QMC module, unused
# # registers 0x12-0x13 NCO phase, unused 

# set_dac_nco(self, channel: int, freq_mhz: float, phase_deg: float = 0)
zynq.dac2.set_dac_nco(1, 00.0, 0)
zynq.dac2.set_dac_nco(2, 00.0, 0)

# PLL Disable:
zynq.dac2.write_register(0x18, 0x0000) # pll_ena=0, 
zynq.dac2.write_register(0x19, 0x0000)
zynq.dac2.write_register(0x1A, 0x0020) # pll_sleep=1

# # PLL Enable:
# zynq.dac2.write_register(0x18, 0x0460)  # pll_ena=1, pll_cp[1:0]=01 single charge pump, pll_p[2:0]=100 (4)
# zynq.dac2.write_register(0x19, 0x0434)  # pll_m[7:0]= 0 000 0100 (4), pll_n[3:0]=0011 (4), pll_vcoitune[1:0]=01
# zynq.dac2.write_register(0x1A, 0xFC00)  # pll_vco[5:0]=1111 11 (4GHz), pll_sleep=0
# lfvolt2 = zynq.dac2.read_register(0x18) # read pll_lfvolt to check for lock
# print(f"DAC2- PLL LOCK : REG0x18 = 0x{lfvolt2:04X}")

zynq.dac2.write_register(0x1B, 0x0800) # fuse_sleep=1

zynq.dac2.write_register(0x1E, 0x8888) # sync_sel for QMC module- SIF_SYNC
zynq.dac2.write_register(0x1F, 0x2224) # syncsel_mixerAB, syncsel_mixerCD, syncsel_nco: OSTR, syncsel_dataformatter: SYNC 
zynq.dac2.write_register(0x20, 0x1400) # syncsel_fifoin: SYNC, syncsel_fifoout: OSTR, clkdiv_sync_sel: OSTR
# ___________________________________________________________________________________________________________________


# ___________________________________________________________________________________________________________________
# xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx DAC1 Initialization- Part 1 xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
zynq.dac1.write_register(0x00, 0x089F) # INTERP = 16, fifo on, clkdiv_sync_ena invsincAB+CD
zynq.dac1.write_register(0x01, 0x050E)
zynq.dac1.write_register(0x02, 0xF052) # 16bit_in, mixer_ena, nco_ena, twos_comp
zynq.dac1.write_register(0x03, 0xF000)
zynq.dac1.write_register(0x07, 0xD8FF) # un-mask FIFO collison, DACCLK-gone, DATACLK-gone
time.sleep(0.1)

# # registers 0x08-0x11 are QMC module, unused
# # registers 0x12-0x13 NCO phase, unused 

zynq.dac1.set_dac_nco(1, 00.0, 0)
zynq.dac1.set_dac_nco(2, 00.0, 0)

# PLL Disable:
zynq.dac1.write_register(0x18, 0x0000) # pll_ena=0, 
zynq.dac1.write_register(0x19, 0x0000)
zynq.dac1.write_register(0x1A, 0x0020) # pll_sleep=1

# # PLL Enable:
# zynq.dac1.write_register(0x18, 0x0460)  # pll_ena=1, pll_cp[1:0]=01 single charge pump, pll_p[2:0]=100 (4)
# zynq.dac1.write_register(0x19, 0x0434)  # pll_m[7:0]= 0 000 0100 (4), pll_n[3:0]=0011 (4), pll_vcoitune[1:0]=01
# zynq.dac1.write_register(0x1A, 0xFC00)  # pll_vco[5:0]=1111 11 (4GHz), pll_sleep=0
# lfvolt2 = zynq.dac1.read_register(0x18) # read pll_lfvolt to check for lock
# print(f"DAC1- PLL LOCK: REG0x18 = 0x{lfvolt2:04X}")

zynq.dac1.write_register(0x1B, 0x0800) # fuse_sleep=1

zynq.dac1.write_register(0x1E, 0x8888) # sync_sel for QMC module- SIF_SYNC
zynq.dac1.write_register(0x1F, 0x2224) # syncsel_mixerAB, syncsel_mixerCD, syncsel_nco: OSTR, syncsel_dataformatter: SYNC 
zynq.dac1.write_register(0x20, 0x1400) # syncsel_fifoin: SYNC, syncsel_fifoout: OSTR, clkdiv_sync_sel: OSTR
# ___________________________________________________________________________________________________________________


# ___________________________________________________________________________________________________________________
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Clock configuration for both DACs ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
## Add FPGACLK in order to start all LVDS data and clocks
# DACCLK1 DACCLK2 1 GHz, OSTR effective 15.625 MHz, FPGA_CLK 250 MHz, ADC_CLK off
CDCE_CONFIG_wFPGA = '8340032083400321EB040302811C0303EB84031410000BE504BE09F6BD0037F7'
zynq.configure_clock(CDCE_CONFIG_wFPGA)
print('CDCE configured with FPGACLK')
time.sleep(0.001)
# ___________________________________________________________________________________________________________________


# ___________________________________________________________________________________________________________________
#--------------------------------------------- DAC2 Initialization- Part 2 -------------------------------------------
## Check pll lock and reset alarms register
lfvolt2 = zynq.dac2.read_register(0x18) # read pll_lfvolt to check for lock
print(f"✓ PLL LOCK: REG0x18 = 0x{lfvolt2:04X}")

zynq.dac2.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC2 = zynq.dac2.read_register(0x05) # read reg5 to check for alarms
print(f"DAC2- Alarms Register: REG0x05 = 0x{reg5_DAC2:04X}")
time.sleep(0.001)

### Disable clock divider sync
zynq.dac2.write_register(0x1F, 0x2224) # turn on sif_sync
zynq.dac2.write_register(0x00, 0x089B) # INTERP = 16, fifo on, clkdiv_sync_ena=0, invsincAB+CD=1
time.sleep(0.001)
zynq.dac2.write_register(0x1F, 0x2228) # turn off sif_sync, syncsel_dataformatter- no sync
zynq.dac2.write_register(0x20, 0x0000) # Disable FIFO input and output pointer sync
zynq.dac2.write_register(0x03, 0xF001) # Set TXENABLE high
zynq.dac2.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC2 = zynq.dac2.read_register(0x05) # read reg5 to check for alarms
print(f"DAC2- Alarms Register : REG0x05 = 0x{reg5_DAC2:04X}")
zynq.dac2.write_register(0x24, 0x1C00)  
# ___________________________________________________________________________________________________________________


# ___________________________________________________________________________________________________________________
# xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx DAC1 Initialization- Part 2 xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
## Check pll lock and reset alarms register
lfvolt1 = zynq.dac1.read_register(0x18) # read pll_lfvolt to check for lock
print(f"DAC1- PLL LOCK: REG0x18 = 0x{lfvolt1:04X}")

zynq.dac1.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC1 = zynq.dac1.read_register(0x05) # read reg5 to check for alarms
print(f"DAC1- Alarms Register: REG0x05 = 0x{reg5_DAC1:04X}")
time.sleep(0.001)

### Disable clock divider sync
zynq.dac1.write_register(0x1F, 0x2224) # turn on sif_sync
zynq.dac1.write_register(0x00, 0x089B) # INTERP = 16, fifo on, clkdiv_sync_ena=0, invsincAB+CD=1
time.sleep(0.001)
zynq.dac1.write_register(0x1F, 0x2228) # turn off sif_sync, syncsel_dataformatter- no sync


zynq.dac1.write_register(0x20, 0x0000) # Disable FIFO input and output pointer sync
zynq.dac1.write_register(0x03, 0xF001) # Set TXENABLE high
zynq.dac1.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC1 = zynq.dac1.read_register(0x05) # read reg5 to check for alarms
print(f"DAC1- Alarms Register: REG0x05 = 0x{reg5_DAC1:04X}")
zynq.dac1.write_register(0x24, 0x1C00) 
# ___________________________________________________________________________________________________________________

DAC2 NCO configured: word=0x00000000
DAC2 NCO configured: word=0x00000000
DAC1 NCO configured: word=0x00000000
DAC1 NCO configured: word=0x00000000
CDCE configured with FPGACLK
✓ PLL LOCK: REG0x18 = 0x0007
DAC2- Alarms Register: REG0x05 = 0x3960
DAC2- Alarms Register : REG0x05 = 0x0860
DAC1- PLL LOCK: REG0x18 = 0x0007
DAC1- Alarms Register: REG0x05 = 0x3978
DAC1- Alarms Register: REG0x05 = 0x1878


**-----------------------------------------------------------------------------------------------------------------------**
## BRAM profile control



In [67]:
zynq.bram_init()

profile0 = [(0.9,  1.0,   0)]   # One tone- full amplitude
profile1 = [(0.3,  1.0,   0), (0.3, 1.1, 0)] # Two tones, Overflow taken into consideration
profile2 = [(0.9,  2.0,   0)]   # One tone- full amplitude
profile3 = [(0.3,  2.0,   0), (0.3, 2.2, 0)] # Two tones, Overflow taken into consideration
profiles = [profile0, profile1, profile2, profile3]
# --- Write to both channels ---
# for ch in (1, 2):
#     for pr in(0,2):
#         zynq.set_freq_profile(channel=ch, profile_num=pr, data=profiles[pr], DEBUG=1)
#         print(f"✓ CH{ch}: 4 frequency profiles written")
zynq.set_freq_profile(channel=1, profile_num=0, data=profile0, DEBUG=1)
zynq.set_freq_profile(channel=1, profile_num=1, data=profile1, DEBUG=1)
zynq.set_freq_profile(channel=1, profile_num=2, data=profile2, DEBUG=1)
zynq.set_freq_profile(channel=1, profile_num=3, data=profile3, DEBUG=1)

zynq.set_freq_profile(channel=2, profile_num=0, data=profile0, DEBUG=1)
zynq.set_freq_profile(channel=2, profile_num=1, data=profile1, DEBUG=1)
zynq.set_freq_profile(channel=2, profile_num=2, data=profile2, DEBUG=1)
zynq.set_freq_profile(channel=2, profile_num=3, data=profile3, DEBUG=1)


print("\nDone.")

NUM_STEPS = 1000 # ramp duration in clock ticks

# --- Define ramp shape (one or more segments) ---
on_ramp_0  = [(NUM_STEPS,  4000)]   # linear rise:  500 steps × +1000
off_ramp_0 = [(NUM_STEPS, -4000)]   # linear fall:  500 steps × -1000

on_ramp_1  = [(NUM_STEPS,  3000)]   # linear rise:  500 steps × +1000
off_ramp_1 = [(NUM_STEPS, -3000)]   # linear fall:  500 steps × -1000

on_ramp_2  = [(500,  1000),(2000,0),(500,2000)]   # linear rise:  500 steps × +1000
off_ramp_2 = [(500,  -1000),(2000,0),(500,-2000)]   # linear fall:  500 steps × -1000

on_ramp_3  = [(1000,  2000),(2000,0),(500,2000)]   # linear rise:  500 steps × +1000
off_ramp_3 = [(500,  -1000),(2000,0),(500,-2000)]   # linear fall:  500 steps × -1000

amp_on_profiles = [on_ramp_0, on_ramp_1, on_ramp_2, on_ramp_3]
amp_off_profiles = [off_ramp_0, off_ramp_1, off_ramp_2, off_ramp_3]

# --- Write to both channels (independent allocators) ---
for ch in (1, 2):
    alloc = zynq.get_amp_allocator(channel=ch)
    for profile_num in range(0,4):     # profiles 0-3, matching freq profiles above
        zynq.set_amp_profile(
            channel=ch,
            profile_num=profile_num,
            on_ramps=amp_on_profiles[profile_num],
            off_ramps=amp_off_profiles[profile_num],
            allocator=alloc,
            DEBUG=1,
        )
    print(f"✓ CH{ch}: amplitude profiles written")
    
    
print("\nDone.")


[BRAM] Initializing...
✓ BRAM initialized (pointer tables cleared)


[FREQ_PROFILE] CH1 (MT_0/IQ_32tones_w_bram/axi_bram_ctrl_0)  Profile 0  (1 signal(s))
  [ 0] amp=0.900  freq=1.0 MHz  phase=0°  → 0x010624DD  0x0000E665
✓ 1 signal(s) written


[FREQ_PROFILE] CH1 (MT_0/IQ_32tones_w_bram/axi_bram_ctrl_0)  Profile 1  (2 signal(s))
  [ 0] amp=0.300  freq=1.0 MHz  phase=0°  → 0x010624DD  0x00004CCC
  [ 1] amp=0.300  freq=1.1 MHz  phase=0°  → 0x01205BC0  0x00004CCC
✓ 2 signal(s) written


[FREQ_PROFILE] CH1 (MT_0/IQ_32tones_w_bram/axi_bram_ctrl_0)  Profile 2  (1 signal(s))
  [ 0] amp=0.900  freq=2.0 MHz  phase=0°  → 0x020C49BA  0x0000E665
✓ 1 signal(s) written


[FREQ_PROFILE] CH1 (MT_0/IQ_32tones_w_bram/axi_bram_ctrl_0)  Profile 3  (2 signal(s))
  [ 0] amp=0.300  freq=2.0 MHz  phase=0°  → 0x020C49BA  0x00004CCC
  [ 1] amp=0.300  freq=2.2 MHz  phase=0°  → 0x0240B780  0x00004CCC
✓ 2 signal(s) written


[FREQ_PROFILE] CH2 (MT_1/IQ_32tones_w_bram/axi_bram_ctrl_0)  Profile 0  (1 signal(s))
  

### Single Tone Control

In [68]:
## Configure DAC input
DDS_CLK_HZ = 62_500_000
from pynq_dds import DAC2_DDS
dds2 = DAC2_DDS(ol, dds_clk_hz=DDS_CLK_HZ)

# Configurable frequency and phase
AB_FREQ = 5.0    # MHz
AB_PHASE = 0.0   # degrees
CD_FREQ = 5.0    # MHz
CD_PHASE = 0.0   # degrees

dds2.set(ab_freq_mhz=AB_FREQ,  ab_phase_deg=AB_PHASE,
         cd_freq_mhz=CD_FREQ,  cd_phase_deg=CD_PHASE)

print(f"\n✓ DAC2 DDS configured ({DDS_CLK_HZ/1e6:.1f} MHz clock):")
print(f"  AB: {AB_FREQ} MHz @ {AB_PHASE}°")
print(f"  CD: {CD_FREQ} MHz @ {CD_PHASE}°")

DAC2 DDS ready  (CLK = 62.5 MHz)
AB: 5.000 MHz  0.0°  PINC=0x147AE147  POFF=0x00000000
CD: 5.000 MHz  0.0°  PINC=0x147AE147  POFF=0x00000000

✓ DAC2 DDS configured (62.5 MHz clock):
  AB: 5.0 MHz @ 0.0°
  CD: 5.0 MHz @ 0.0°


In [15]:
dds2.off()

AB: 0.000 MHz  0.0°  PINC=0x00000000  POFF=0x00000000
CD: 0.000 MHz  0.0°  PINC=0x00000000  POFF=0x00000000
DAC2 DDS off


## ---
## Diagnostic / debug cells

In [17]:
# Read back all CDCE registers
regs = zynq.cdce.read_all()

CDCE62005 Register Dump:
  reg[0] = 0x8340032
  reg[1] = 0x8340032
  reg[2] = 0xEB04030
  reg[3] = 0x811C030
  reg[4] = 0xEB84031
  reg[5] = 0x10000BE
  reg[6] = 0x04BE09F
  reg[7] = 0xBD0037F
  reg[8] = 0x80009BF


In [17]:
# Read back all DAC registers
# zynq.read_all_dac_registers(dac_num=1)
zynq.read_all_dac_registers(dac_num=2)


[DAC2] Reading all registers:
  0x00: 0x089B
  0x01: 0x050E
  0x02: 0xF052
  0x03: 0xF001
  0x04: 0xDA4C
  0x05: 0x3978
  0x06: 0x3300
  0x07: 0xD8FF
  0x08: 0x0000
  0x09: 0x8000
  0x0A: 0x0000
  0x0B: 0x0000
  0x0C: 0x0400
  0x0D: 0x0400
  0x0E: 0x0400
  0x0F: 0x0400
  0x10: 0x0000
  0x11: 0x0000
  0x12: 0x0000
  0x13: 0x0000
  0x14: 0x0000
  0x15: 0x0000
  0x16: 0x0000
  0x17: 0x0000
  0x18: 0x0466
  0x19: 0x0434
  0x1A: 0xFC00
  0x1B: 0x0800
  0x1C: 0x0000
  0x1D: 0x0000
  0x1E: 0x8888
  0x1F: 0x2228
  0x20: 0x0000
  0x21: 0x0000
  0x22: 0x1B1B
  0x23: 0xFFFF
  0x24: 0x1800
  0x25: 0x7A7A
  0x26: 0xB6B6
  0x27: 0xEAEA
  0x28: 0x4545
  0x29: 0x1A1A
  0x2A: 0x1616
  0x2B: 0xAAAA
  0x2C: 0xC6C6
  0x2D: 0x0004
  0x2E: 0x0000
  0x2F: 0x0000
  0x30: 0x0000
  0x7F: 0x540C


{0: 2203,
 1: 1294,
 2: 61522,
 3: 61441,
 4: 55884,
 5: 14712,
 6: 13056,
 7: 55551,
 8: 0,
 9: 32768,
 10: 0,
 11: 0,
 12: 1024,
 13: 1024,
 14: 1024,
 15: 1024,
 16: 0,
 17: 0,
 18: 0,
 19: 0,
 20: 0,
 21: 0,
 22: 0,
 23: 0,
 24: 1126,
 25: 1076,
 26: 64512,
 27: 2048,
 28: 0,
 29: 0,
 30: 34952,
 31: 8744,
 32: 0,
 33: 0,
 34: 6939,
 35: 65535,
 36: 6144,
 37: 31354,
 38: 46774,
 39: 60138,
 40: 17733,
 41: 6682,
 42: 5654,
 43: 43690,
 44: 50886,
 45: 4,
 46: 0,
 47: 0,
 48: 0,
 127: 21516}

In [ ]:
# Read back key ADC registers
zynq.adc_read_all_registers()

### BRAM diagnostics

In [19]:
# Read back a frequency profile from BRAM and print it
CH      = 1     # 1 = DAC1 AB (MT_0),  2 = DAC1 CD (MT_1)
PROFILE = 1     # 0-7

profile_data = zynq.get_freq_profile(channel=CH, profile_num=PROFILE)
if profile_data:
    print(f"\nFrequency profile CH{CH} / profile {PROFILE}:")
    for i, (amp, freq, phase) in enumerate(profile_data):
        print(f"  [{i}]  amp={amp:.3f}  freq={freq:.3f} MHz  phase={phase:.1f}°")


[FREQ_PROFILE] Reading CH1 Profile 1
  [ 0]  amp=0.500  freq=1.000 MHz  phase=0.0°
  [ 1]  amp=0.500  freq=1.100 MHz  phase=0.0°
  End of profile at signal 2
✓ Read 2 signal(s)


Frequency profile CH1 / profile 1:
  [0]  amp=0.500  freq=1.000 MHz  phase=0.0°
  [1]  amp=0.500  freq=1.100 MHz  phase=0.0°


In [ ]:
# Dump the amplitude BRAM pointer table for one channel
# Shows ON/OFF ramp address for each profile
zynq.bram.dump_pointer_table(channel=1)   # 1=MT_0/AB,  2=MT_1/CD

In [ ]:
# Print full memory allocation map for a channel's amplitude allocator
zynq.get_amp_allocator(channel=1).print_allocation_map()   # 1=MT_0/AB,  2=MT_1/CD

In [ ]:
# Raw BRAM word dump — cross-check against ILA / VHDL simulation
CH     = 1       # 1=MT_0/AB,  2=MT_1/CD
OFFSET = 0x000   # 0x000 = start of profile 0;  0x100 = start of profile 1

raw = zynq.bram.read_freq_bram_raw(channel=CH, n_words=8, byte_offset=OFFSET)
print(f"Freq BRAM CH{CH} ({zynq.bram.FREQ_MEM_NAMES[CH]}) @ 0x{OFFSET:04X}:")
for i, w in enumerate(raw):
    print(f"  [0x{OFFSET + i*4:04X}]  0x{w:08X}")